# Global Propensity Score Model
Fit and save a global propensity score model. This model predicts the likelihood of a 1km2 pixel of being protected based on a set of covariates: elevation, slope, tree cover in 2000, travel time to the nearest densely-populated area, and population density. The model is fitted independently for each biome, since covariates have different relationships with protection status depending on biome.

In [ ]:
from pathlib import Path
import sys
import os
import pandas as pd
import ee
import json
import geemap
import numpy as np
from pyproj import Transformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    PAS_ASSET_ID,
    OECMS_ASSET_ID,
    HGFC_ASSET_ID,
    BIOME_ASSET_ID,
    EE_CRS_METERS,
    STRATA_ASSET_ID,
    PSM_CELL_SIZE,
    GCS_BUCKET,
    PSM_SAMPLES_PREFIX,
    STATE_FILE,
    TOTAL_POINTS,
    TREAT_CONTROL,
    MIN_PER_STRATUM,
    COVARIATES
)

from psm.tiling import tiles_to_feature_collection
from psm.tiling import build_tile_grid, filter_tiles_to_land
from psm.allocation import (
    compute_global_allocation,
    compute_tile_pixel_counts,
    compute_tile_allocations,
    validate_allocation,
)
from psm.sampling import build_sample_export_task, TileTaskManager
from psm.covariates import build_resampled_covariates

ee.Initialize(project=PROJECT)

EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

## Data Prep
Load and resample covariates and layers for sample stratification.

In [ ]:
# Make a binary PA mask (protected = 1, unprotected = 0)

PAs = ee.FeatureCollection(PAS_ASSET_ID)
OECMS = ee.FeatureCollection(OECMS_ASSET_ID)
all_PAs = (
    ee.FeatureCollection([PAs, OECMS])
    .flatten()
    .filter(ee.Filter.eq("REALM", "Terrestrial"))
)
protected_mask = (
    ee.Image.constant(0)
    .rename("protected")
    .paint(featureCollection=all_PAs, color=1)
    .reproject(crs=EE_CRS_1km)
)

# Build and resample covariate layers

covariate_bands = build_resampled_covariates(EE_CRS_1km)
covariates = protected_mask.addBands(covariate_bands)

# Make biome layer and combine it with PA binary to make a strata band.
# Each stratum is a unique combo of biome and PA status:
# e.g. 6 = unprotected boreal forest, 34 = protected mangroves

biome_fc = ee.FeatureCollection(BIOME_ASSET_ID).map(
    lambda f: f.set("BIOME_NUM", ee.Number(f.get("BIOME_NUM")).int())
)
biome = (
    ee.Image(0)
    .paint(featureCollection=biome_fc, color="BIOME_NUM")
    .rename("biome")
    .reproject(protected_mask.projection())
    .toInt()
)
strata = protected_mask.multiply(20).add(biome).rename("strata").toInt()
covariates = covariates.addBands(strata)

# Apply land mask to avoid sampling oceans and large permanent water bodies

land_mask = (
    ee.Image(HGFC_ASSET_ID)
    .select("datamask")
    .eq(1)  # 1 = land, 2 = permanent water/ocean, 0 = no data
)
covariates = covariates.updateMask(land_mask)

print("Covariate bands:", covariates.bandNames().getInfo())

In [ ]:
# Export the strata image to asset

strata_export_task = ee.batch.Export.image.toAsset(
    image=strata,
    description="strata_1km_export",
    assetId=STRATA_ASSET_ID,
    region=ee.Geometry.BBox(-180, -60, 180, 84),
    scale=PSM_CELL_SIZE,
    crs=EE_CRS_METERS,
    maxPixels=1e13,
)
# strata_export_task.start()
# print("Strata export started:", strata_export_task.id)

In [ ]:
# Visualize covariates

from utils.variables import BIOME_PALETTE

Map = geemap.Map()
Map.add_basemap("CartoDB.Positron")
Map.setCenter(-1.0232, 7.9465, 6)

Map.addLayer(
    covariates.select("protected"),
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "Protected"
)
Map.addLayer(
    covariates.select("elevation"),
    {"min": 0, "max": 800, "palette": ["blue", "green", "yellow", "red"]},
    "Elevation", 0
)
Map.addLayer(
    covariates.select("slope"),
    {"min": 0, "max": 12, "palette": ["blue", "green", "yellow", "red"]},
    "Slope", 0
)
Map.addLayer(
    covariates.select("treecover2000"),
    {"min": 0, "max": 100, "palette": ["white", "green"]},
    "Tree Cover 2000", 0
)
Map.addLayer(
    covariates.select("travel_time"),
    {"min": 0, "max": 1200, "palette": ["blue", "green", "yellow", "red"]},
    "Travel Time", 0
)
Map.addLayer(
    covariates.select("log_pop_density"),
    {"min": 0, "max": 6, "palette": ["blue", "green", "yellow", "red"]},
    "Population Density", 0
)
Map.addLayer(
    covariates.select("human_footprint"),
    {"min": 0, "max": 35, "palette": ["blue", "green", "yellow", "red"]},
    "Human Footprint", 0
)
Map.addLayer(
    covariates.select("ag_suitability"),
    {"min": 0, "max": 9423, "palette": ["blue", "green", "yellow", "red"]},
    "Agricultural Suitability", 0
)
Map.addLayer(biome.updateMask(land_mask), {"min": 1, "max": 14, "palette": BIOME_PALETTE}, "Biome", 0)
Map.addLayer(covariates.select("strata"), {"min": 1, "max": 34, "palette": ["white", "black"]}, "Strata", 0)

Map

## Stratified Sample Allocation
Stratify samples by biome and protection status, computing allocations per tile.

In [ ]:
# Divide the globe into tiles and filter to tiles that intersect land

tiles_all = build_tile_grid(tile_size_deg=20.0)
# print(f"Total candidate tiles: {len(tiles_all)}")
# 144 total tiles

tiles = filter_tiles_to_land(tiles_all, coarse_scale=10_000, min_land_fraction=0.001)
# print(f"Land tiles: {len(tiles)}")
# 86 land tiles

In [ ]:
# Check that tile grid looks correct

Map = geemap.Map()
Map.add_basemap("CartoDB.Positron")
Map.addLayer(tiles_to_feature_collection(tiles), {"color": "red"}, "Land tiles")

Map.setCenter(0, 20, 2)
Map

In [ ]:
# Globally allocate samples.
# Equal number of samples for each biome, with a 2:1 ratio of unprotected to protected.

global_allocation = compute_global_allocation(
    total_points=TOTAL_POINTS,
    treat_control_ratio=TREAT_CONTROL,
    min_per_stratum=MIN_PER_STRATUM,
)

# Check that global allocation worked as expected

alloc_df = pd.DataFrame([
    {"stratum_id": k, "protected": k // 20, "biome": k % 20, "n": v}
    for k, v in global_allocation.items()
]).sort_values("stratum_id")
print(alloc_df)
print(f"\nTotal allocated: {alloc_df['n'].sum():,}")
print(f"Target: {TOTAL_POINTS:,}")

In [ ]:
# Calculate the number of pixels per stratum in each tile

strata_asset = ee.Image(STRATA_ASSET_ID)

tile_pixel_counts = compute_tile_pixel_counts(
    tiles=tiles,
    strata_image=strata_asset,
    scale=PSM_CELL_SIZE,
    projection=ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE),
    max_workers=10,
)

# Check that pixel-counting worked as expected

for tile_id in list(tile_pixel_counts.keys())[:5]:
    counts = tile_pixel_counts[tile_id]
    print(f"\nTile {tile_id}: {len(counts)} strata, {sum(counts.values()):,} total pixels")
    print(counts)

In [ ]:
# Allocate samples to strata within each tile based on pixel counts.

tile_allocations = compute_tile_allocations(
    tile_pixel_counts=tile_pixel_counts,
    global_allocation=global_allocation,
    min_tile_stratum=1,
)

# Check that per-tile sample allocation worked as expected

# Validate: per-stratum sum across tiles should match global budget
validation_df = validate_allocation(tile_allocations, global_allocation, tolerance=0.02)
print(validation_df)

# Per-tile inspection
print("\nPer-tile sample counts (first 10):")
for tile_id in list(tile_allocations.keys())[:10]:
    alloc = tile_allocations[tile_id]
    n_total = sum(alloc.values())
    n_strata = len(alloc)
    print(f"  {tile_id}: {n_total:,} samples across {n_strata} strata")

# Overall
total_samples = sum(sum(a.values()) for a in tile_allocations.values())
print(f"\nGrand total: {total_samples:,}")

## Sampling
Extract covariate values at sample point locations, one tile at a time.
Save samples to GCS bucket.

In [ ]:
# Test on a single tile first

test_tile_id = "10_01"
test_tile_geom = tiles[test_tile_id]
test_alloc = tile_allocations[test_tile_id]

print(f"Test tile: {test_tile_id}")
print(f"  Samples requested: {sum(test_alloc.values()):,}")
print(f"  Strata: {len(test_alloc)}")

test_task, n_req = build_sample_export_task(
    tile_id=test_tile_id,
    tile_geom=test_tile_geom,
    tile_allocation=test_alloc,
    covariates=covariates,
    strata_image=strata_asset,
    sample_bands=COVARIATES + ["protected"],
    bucket=GCS_BUCKET,
    file_prefix=PSM_SAMPLES_PREFIX,
    scale=PSM_CELL_SIZE,
    projection=EE_CRS_1km,
)
test_task.start()
print(f"Task started: {test_task.id}")

import time
for _ in range(60):
    state = test_task.status().get("state")
    print(state)
    if state in {"COMPLETED", "FAILED"}:
        break
    time.sleep(10)
print(test_task.status())

In [ ]:
# Check that the test tile worked as expected

import pandas as pd

test_df = pd.read_csv(f"gs://{GCS_BUCKET}/{PSM_SAMPLES_PREFIX}{test_tile_id}.csv")
print(f"Rows: {len(test_df):,}  (requested: {n_req:,})")
print(f"Protected: {test_df['protected'].value_counts().to_dict()}")
print(f"Missing: {test_df.isna().sum().sum()}")
print(f"\nCovariate summaries:")
print(test_df[COVARIATES].describe())

In [ ]:
# Run the remaining 85 tiles

Path(STATE_FILE).parent.mkdir(parents=True, exist_ok=True)
manager = TileTaskManager(STATE_FILE)
sample_bands = COVARIATES + ["protected"]

# Register the test tile as already complete
manager.register(
    tile_id=test_tile_id,
    n_requested=sum(test_alloc.values()),
    gcs_path=f"gs://{GCS_BUCKET}/{PSM_SAMPLES_PREFIX}{test_tile_id}.csv",
)
manager.state[test_tile_id].task_id = test_task.id
manager.state[test_tile_id].status = "COMPLETED"
manager.state[test_tile_id].attempts = 1
manager._save()

# Launch all remaining tiles
n_launched = 0
n_skipped = 0
for tile_id, tile_geom in tiles.items():
    alloc = tile_allocations.get(tile_id, {})
    if not alloc:
        n_skipped += 1
        continue

    n_requested = sum(alloc.values())
    gcs_path = f"gs://{GCS_BUCKET}/{PSM_SAMPLES_PREFIX}{tile_id}.csv"

    state = manager.register(tile_id, n_requested, gcs_path)
    if state.status == "COMPLETED":
        n_skipped += 1
        continue
    if state.status == "RUNNING":
        n_skipped += 1
        continue

    task, _ = build_sample_export_task(
        tile_id=tile_id,
        tile_geom=tile_geom,
        tile_allocation=alloc,
        covariates=covariates,
        strata_image=strata_asset,
        sample_bands=sample_bands,
        bucket=GCS_BUCKET,
        file_prefix=PSM_SAMPLES_PREFIX,
        scale=PSM_CELL_SIZE,
        projection=EE_CRS_1km,
    )
    manager.submit(tile_id, task)
    n_launched += 1

print(f"Launched: {n_launched}")
print(f"Skipped (already complete): {n_skipped}")
print(f"Total registered: {len(manager.state)}")

In [ ]:
# Show failed tile details
failed = [tid for tid, st in manager.state.items() if st.status == "FAILED"]
print(f"\nFailed tiles: {failed}")
for tile_id in failed:
    state = manager.state[tile_id]
    print(f"  {tile_id}: {state.error} (requested: {state.n_requested:,})")

In [ ]:
# Divide failed tiles into smaller sub-tiles and retry

import ee

failed_ids = ["03_05", "05_05", "12_05"]

# Mark parent tiles as split so we don't retry them whole
for pid in failed_ids:
    manager.state[pid].status = "SPLIT"
manager._save()

# For each failed tile, split into 4 quadrants and launch each with 1/4 the allocation
for parent_id in failed_ids:
    parent_geom = tiles[parent_id]
    parent_alloc = tile_allocations[parent_id]
    bounds = parent_geom.bounds().coordinates().get(0).getInfo()
    lon_min, lat_min = bounds[0]
    lon_max, lat_max = bounds[2]
    lon_mid = (lon_min + lon_max) / 2
    lat_mid = (lat_min + lat_max) / 2

    quadrants = {
        f"{parent_id}_sw": [lon_min, lat_min, lon_mid, lat_mid],
        f"{parent_id}_se": [lon_mid, lat_min, lon_max, lat_mid],
        f"{parent_id}_nw": [lon_min, lat_mid, lon_mid, lat_max],
        f"{parent_id}_ne": [lon_mid, lat_mid, lon_max, lat_max],
    }

    # Allocate 1/4 of each stratum's points to each quadrant (good enough approximation)
    sub_alloc = {s: max(1, n // 4) for s, n in parent_alloc.items()}

    for sub_id, coords in quadrants.items():
        sub_geom = ee.Geometry.Rectangle(coords, proj="EPSG:4326", geodesic=False)
        task, _ = build_sample_export_task(
            tile_id=sub_id,
            tile_geom=sub_geom,
            tile_allocation=sub_alloc,
            covariates=covariates,
            strata_image=strata_asset,
            sample_bands=sample_bands,
            bucket=GCS_BUCKET,
            file_prefix=PSM_SAMPLES_PREFIX,
            scale=PSM_CELL_SIZE,
            projection=EE_CRS_1km,
        )
        manager.register(sub_id, sum(sub_alloc.values()), f"gs://{GCS_BUCKET}/{PSM_SAMPLES_PREFIX}{sub_id}.csv")
        manager.submit(sub_id, task)
        print(f"Launched {sub_id}")

In [ ]:
# Load and concatenate all samples

from google.cloud import storage

client = storage.Client()
blobs = [
    blob
    for blob in client.list_blobs(GCS_BUCKET, prefix=PSM_SAMPLES_PREFIX)
    if blob.name.endswith(".csv")
]
print(f"Total CSVs in bucket: {len(blobs)}")

dfs = []
for blob in blobs:
    df = pd.read_csv(f"gs://{GCS_BUCKET}/{blob.name}")
    dfs.append(df)

samples = pd.concat(dfs, ignore_index=True)
print(f"Total samples: {len(samples):,}  (target: 100,000)")
print(f"Protected: {samples['protected'].value_counts().to_dict()}")
print(f"Tiles: {samples['tile_id'].nunique()}")
print(f"\nPer-stratum counts (sorted):")
print(samples["strata"].value_counts().sort_index())

In [ ]:
# Remove samples that ended up in non-biome areas
# (Minor edge effect from RESOLVE dataset)

samples_clean = samples[~samples["strata"].isin([0, 20])].reset_index(drop=True)
print(f"Before: {len(samples):,}")
print(f"After: {len(samples_clean):,}")
print(f"Dropped: {len(samples) - len(samples_clean)}")

In [ ]:
# Thin samples within each stratum so they are at least 3km apart to avoid spatial autocorrelation

# Parse the .geo column into lon, lat columns
samples_clean["lon"] = samples_clean[".geo"].apply(lambda s: json.loads(s)["coordinates"][0])
samples_clean["lat"] = samples_clean[".geo"].apply(lambda s: json.loads(s)["coordinates"][1])

# Project to EPSG:6933 (equal-area meters)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True)
x, y = transformer.transform(samples_clean["lon"].values, samples_clean["lat"].values)
samples_clean["x_m"] = x
samples_clean["y_m"] = y

# Snap to 3 km grid cells
THIN_DISTANCE = 3000
samples_clean["_cell_x"] = (samples_clean["x_m"] // THIN_DISTANCE).astype(np.int64)
samples_clean["_cell_y"] = (samples_clean["y_m"] // THIN_DISTANCE).astype(np.int64)

# Stratify thinning by stratum: keep one sample per (cell_x, cell_y, strata)
# This preserves stratum balance even if a cell has both protected and unprotected samples
samples_thinned = (
    samples_clean
    .sample(frac=1, random_state=42)  # shuffle so duplicate retention is random
    .drop_duplicates(["_cell_x", "_cell_y", "strata"], keep="first")
    .drop(columns=["_cell_x", "_cell_y"])
    .reset_index(drop=True)
)

# Save the thinned samples
samples_thinned.to_parquet("data/samples_thinned.parquet")

print(f"Before thinning: {len(samples_clean):,}")
print(f"After thinning:  {len(samples_thinned):,}")
print(f"Removed: {len(samples_clean) - len(samples_thinned):,} ({(1 - len(samples_thinned)/len(samples_clean)):.1%})")

print(f"\nPer-stratum counts after thinning:")
print(samples_thinned["strata"].value_counts().sort_index())

In [ ]:
# Visualize samples for one tile

TILE = "09_03"
d = samples_thinned.loc[samples_thinned["tile_id"] == TILE]

fc = ee.FeatureCollection(
    d.apply(
        lambda r: ee.Feature(
            ee.Geometry.Point([r["lon"], r["lat"]]),
            {"protected": int(r["protected"])},
        ),
        axis=1,
    ).tolist()
)

Map_tile = geemap.Map()
Map_tile.add_basemap("CartoDB.Positron")
Map_tile.addLayer(
    tiles_to_feature_collection({TILE: tiles[TILE]}),
    {"color": "yellow", "fillColor": "00000000"},
    f"tile {TILE}",
)
Map_tile.addLayer(fc.filter(ee.Filter.eq("protected", 0)), {"color": "0000ff"}, "unprotected")
Map_tile.addLayer(fc.filter(ee.Filter.eq("protected", 1)), {"color": "ff0000"}, "protected")
Map_tile.centerObject(fc, 5)
Map_tile

## Model Fitting
Use sampled training data to fit a propensity model. This model calculates biome-specific coefficients for each covariate, because the effect of each covariate depends on the biome.

In [ ]:
samples_thinned = pd.read_parquet("data/samples_thinned.parquet")

In [ ]:
# Fit model using all training data

X_raw = samples_thinned[COVARIATES].values
y = samples_thinned["protected"].values
biome = samples_thinned["strata"].values % 20

n_main = len(COVARIATES)

# Standardize covariates
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# One-hot encode biome (drop first as reference)
biome_dummies = pd.get_dummies(biome, prefix="biome", drop_first=True).astype(float).values

# Build interaction terms: each covariate × each biome dummy
interactions = []
interaction_names = []
biome_levels = sorted(set(biome))[1:]  # skip reference biome
for i, b in enumerate(biome_levels):
    for j, cov in enumerate(COVARIATES):
        interactions.append(X_scaled[:, j] * biome_dummies[:, i])
        interaction_names.append(f"{cov}_x_biome{b}")
interactions = np.column_stack(interactions)

# Combine: main covariates + biome main effects + interactions
X_full = np.hstack([X_scaled, biome_dummies, interactions])
feature_names = (
    COVARIATES
    + [f"biome{b}" for b in biome_levels]
    + interaction_names
)
# print(f"Design matrix: {X_full.shape}  ({len(feature_names)} features)")

# Fit
model3 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
    random_state=42,
)
model3.fit(X_full, y)

In [ ]:
# Inspect model coefficients

main_coefs = pd.DataFrame({
    "covariate": COVARIATES,
    "coef_in_ref_biome": model3.coef_[0][:n_main],
})
print(f"\nMain covariate effects (in reference biome, biome={sorted(set(biome))[0]}):")
print(main_coefs.to_string(index=False))

print(f"\nPer-biome effective slopes (covariate effect within each biome):")

n_biome = len(biome_levels)
ref_biome = sorted(set(biome))[0]

per_biome_slopes = {ref_biome: model3.coef_[0][:n_main].tolist()}
for i, b in enumerate(biome_levels):
    interaction_start = n_main + n_biome + i * n_main
    interaction_end = interaction_start + n_main
    interaction_coefs = model3.coef_[0][interaction_start:interaction_end]
    per_biome_slopes[b] = (model3.coef_[0][:n_main] + interaction_coefs).tolist()

slope_df = pd.DataFrame(per_biome_slopes, index=COVARIATES).T
slope_df.index.name = "biome"
print(slope_df.round(3))

In [ ]:
# Calculate model AUC using k-fold cross-validation.
# 2 methods: random split and spatial split

from sklearn.model_selection import StratifiedKFold

# Training AUC (in-sample — for diagnostic only, expected to be slightly optimistic)
y_pred_train = model3.predict_proba(X_full)[:, 1]
training_auc = roc_auc_score(y, y_pred_train)
print(f"Training AUC (in-sample):    {training_auc:.4f}  (slightly optimistic — for diagnostic only)")

# Random K-fold CV: standard out-of-sample AUC
# Stratified to maintain protected:unprotected ratio in each fold
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

cv_aucs = []
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_full, y), 1):
    X_train_fold, X_test_fold = X_full[train_idx], X_full[test_idx]
    y_train_fold, y_test_fold = y[train_idx], y[test_idx]

    fold_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )
    fold_model.fit(X_train_fold, y_train_fold)

    y_pred_test = fold_model.predict_proba(X_test_fold)[:, 1]
    fold_auc = roc_auc_score(y_test_fold, y_pred_test)
    cv_aucs.append(fold_auc)
    print(f"  Fold {fold_idx}: {fold_auc:.4f}  (n_train={len(train_idx):,}, n_test={len(test_idx):,})")

cv_auc_mean = np.mean(cv_aucs)
cv_auc_std = np.std(cv_aucs)
print(f"\n{N_FOLDS}-fold CV AUC (random):     {cv_auc_mean:.4f} ± {cv_auc_std:.4f}")

# Spatial K-fold CV: out-of-sample AUC accounting for spatial autocorrelation
# Folds are defined by biome × continent strata to enforce spatial separation
# between train and test sets.
# Approximate continent using a coarse longitude/latitude grid.
samples_for_spatial = samples_thinned.copy()
samples_for_spatial["biome_for_fold"] = samples_for_spatial["strata"].values % 20

# Build coarse spatial bins from lon/lat (20° bins → ~9 longitude bins, ~9 latitude bins)
# This creates fold groups that span large geographic regions
if "lon" not in samples_for_spatial.columns:
    samples_for_spatial["lon"] = samples_for_spatial[".geo"].apply(
        lambda s: json.loads(s)["coordinates"][0]
    )
    samples_for_spatial["lat"] = samples_for_spatial[".geo"].apply(
        lambda s: json.loads(s)["coordinates"][1]
    )

samples_for_spatial["lon_bin"] = (samples_for_spatial["lon"] // 40).astype(int)  # ~40° bins
samples_for_spatial["lat_bin"] = (samples_for_spatial["lat"] // 40).astype(int)
samples_for_spatial["spatial_group"] = (
    samples_for_spatial["lon_bin"].astype(str) + "_" + samples_for_spatial["lat_bin"].astype(str)
)
spatial_groups = samples_for_spatial["spatial_group"].values

# Manually create K spatial folds by assigning each spatial group to a fold
unique_groups = sorted(samples_for_spatial["spatial_group"].unique())
np.random.seed(42)
shuffled_groups = np.random.permutation(unique_groups)
group_to_fold = {g: i % N_FOLDS for i, g in enumerate(shuffled_groups)}
fold_assignments = np.array([group_to_fold[g] for g in spatial_groups])

spatial_aucs = []
for fold_idx in range(N_FOLDS):
    test_mask = fold_assignments == fold_idx
    train_mask = ~test_mask

    if test_mask.sum() < 100 or y[test_mask].sum() == 0 or (y[test_mask] == 0).sum() == 0:
        # Skip folds that are too small or lack class diversity
        print(f"  Spatial fold {fold_idx + 1}: skipped (insufficient data or class diversity)")
        continue

    X_train_s, X_test_s = X_full[train_mask], X_full[test_mask]
    y_train_s, y_test_s = y[train_mask], y[test_mask]

    fold_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )
    fold_model.fit(X_train_s, y_train_s)

    y_pred_s = fold_model.predict_proba(X_test_s)[:, 1]
    fold_auc = roc_auc_score(y_test_s, y_pred_s)
    spatial_aucs.append(fold_auc)
    print(f"  Spatial fold {fold_idx + 1}: {fold_auc:.4f}  (n_train={train_mask.sum():,}, n_test={test_mask.sum():,})")

if spatial_aucs:
    spatial_auc_mean = np.mean(spatial_aucs)
    spatial_auc_std = np.std(spatial_aucs)
    print(f"\nSpatial CV AUC (geographic): {spatial_auc_mean:.4f} ± {spatial_auc_std:.4f}")

# Final reporting
print(f"\n{'='*60}")
print(f"AUC Summary:")
print(f"  Training (in-sample):  {training_auc:.4f}")
print(f"  Random CV (5-fold):    {cv_auc_mean:.4f} ± {cv_auc_std:.4f}")
if spatial_aucs:
    print(f"  Spatial CV (5-fold):   {spatial_auc_mean:.4f} ± {spatial_auc_std:.4f}")
print(f"{'='*60}")

# Use the random CV AUC as the primary metric for saving and reporting
auc = cv_auc_mean

In [ ]:
# Save the model

import joblib
from pathlib import Path
from datetime import datetime

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

artifacts = {
    "model": model3,
    "scaler": scaler,
    "covariates": COVARIATES,
    "biome_levels_ref": sorted(set(biome))[0],
    "biome_levels_dummied": biome_levels,
    "feature_names": feature_names,
    "training_metadata": {
        "n_samples": len(y),
        "training_auc": float(training_auc),
        "cv_auc_random_mean": float(cv_auc_mean),
        "cv_auc_random_std": float(cv_auc_std),
        "cv_auc_spatial_mean": float(spatial_auc_mean) if spatial_aucs else None,
        "cv_auc_spatial_std": float(spatial_auc_std) if spatial_aucs else None,
        "protected_fraction": float(y.mean()),
        "timestamp": timestamp,
        "specification": "Option 3: covariates × biome interactions",
    },
}
joblib.dump(artifacts, model_dir / f"propensity_model_{timestamp}.pkl")
print(f"Saved: models/propensity_model_{timestamp}.pkl")